<a id="curation-lab"></a>
# VideoDB Curation: QA a Dataset from the Curation Lab

The **Curation Lab** applies known problems to a LeRobot v3 dataset in Azure Blob, in place. This
notebook connects that dataset to Curation, then runs **validate** and **audit** and shows what
Curation found as one short table.

Connect once. After each scenario you apply in the Lab, re-run **Validate** and **Audit**.

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/guides/curation/curation_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a id="setup"></a>
## 1. Set up

Set `VIDEO_DB_BASE_URL` to a deployment with Curation enabled, and enter your API key when prompted.
The SDK does not wrap the dataset endpoints yet, so the cells call them with `conn.get` and `conn.post`.

In [ ]:
!pip install -q "videodb>=0.5.0" python-dotenv pandas ipywidgets

In [ ]:
import os
import time
from fractions import Fraction
from getpass import getpass

import pandas as pd
from dotenv import load_dotenv
from videodb import connect
from videodb.exceptions import InvalidRequestError

load_dotenv()
api_key = os.getenv("VIDEO_DB_API_KEY") or getpass("Enter your VideoDB API key: ")
base_url = os.getenv("VIDEO_DB_BASE_URL") or input("VideoDB endpoint (VIDEO_DB_BASE_URL): ")
conn = connect(api_key=api_key, base_url=base_url)
pd.set_option("display.max_colwidth", 90)

Curation answers in detailed JSON. These helpers turn it into one line for validation and one
table of findings for an audit: a row per problem, saying which check, where, and when.

In [ ]:
from fractions import Fraction

PASSING = {"passed", "not_applicable"}
ICON = {"passed": "✅", "flagged": "🚩", "insufficient": "❔", "error": "❌", "not_applicable": "—"}


def _seconds(value) -> str:
    return f"{float(Fraction(str(value))):.2f} s"


def _detail(issue: dict) -> str:
    """One short line saying where the problem is, never the raw JSON."""
    if issue.get("intervals"):
        spans = [f"{_seconds(i['start_seconds'])}–{_seconds(i['end_seconds'])}" for i in issue["intervals"][:3]]
        more = issue.get("intervals_total", len(issue["intervals"])) - len(spans)
        return ", ".join(spans) + (f" (+{more} more)" if more > 0 else "")
    if issue.get("events"):
        times = [f"at {_seconds(e['time_seconds'])} ({e.get('pattern', '')})" for e in issue["events"][:3]]
        more = issue.get("events_total", len(issue["events"])) - len(times)
        return ", ".join(times) + (f" (+{more} more)" if more > 0 else "")
    if issue.get("points"):
        def point(p):
            value = p.get("value")
            return f"row {p['row']} {p['coordinate']}" + (f" = {value}" if value is not None else "")
        points = [point(p) for p in issue["points"][:3]]
        more = issue.get("count", len(issue["points"])) - len(points)
        return ", ".join(points) + (f" (+{more} more)" if more > 0 else "")
    if "observed" in issue and "expected" in issue:
        return f"{issue['observed']} found, {issue['expected']} expected"
    return issue.get("message", "")


def _issues(result: dict):
    """(where, issue) for every issue in a check result, at any depth Curation reports."""
    for issue in result.get("issues", []):
        yield "", issue
    for group in ("cameras", "sequences", "fields"):
        for name, part in (result.get(group) or {}).items():
            for issue in part.get("issues", []):
                yield name.replace("observation.images.", ""), issue


def findings(episodes: list[dict]) -> pd.DataFrame:
    rows = []
    for e in episodes:
        for check, result in e["checks"].items():
            if result["status"] in PASSING:
                continue
            for where, issue in _issues(result):
                status = issue.get("status", result["status"])
                if status in PASSING:
                    continue
                rows.append({"episode": e["episode"], "check": check, "where": where,
                             "result": f"{ICON.get(status, '')} {status}", "flag": issue["code"],
                             "detail": _detail(issue)})
    return pd.DataFrame(rows, columns=["episode", "check", "where", "result", "flag", "detail"])


def summarize_audit(episodes: list[dict], checks: list[str]) -> None:
    review = [e for e in episodes if e["status"] not in PASSING]
    print(f"Audited {len(episodes)} episodes × {len(checks)} checks: "
          f"{len(episodes) - len(review)} pass, {len(review)} need a look.")
    table = findings(episodes)
    if table.empty:
        print("✅ Nothing to review.")
    else:
        display(table.set_index(["episode", "check"]))


def summarize_validation(response=None, error=None) -> None:
    if error is None:
        s = (response or {}).get("summary", response or {})
        cams = ", ".join(c.replace("observation.images.", "") for c in s.get("cameras", []))
        print(f"✅ Valid {s.get('layout', '')} {s.get('codebase_version', '')} · {s.get('episodes', '?')} episodes"
              f" · cameras: {cams or '?'}")
        return
    body = error.response.json()
    problems = (body.get("detail") or body.get("details") or {}).get("errors") or []
    print(f"❌ {body.get('code', 'invalid')}: {body.get('message', '')}")
    if problems:
        display(pd.DataFrame([{"flag": p.get("code"), "file": p.get("path"), "episode": p.get("episode", ""),
                               "detail": p.get("message")} for p in problems]))

<a id="dataset"></a>
## 2. The dataset

In the Lab, click **Connect** and paste its cell over this one. `SAS_SECRET_NAME` is the name of
a Key Vault secret in your VideoDB deployment that holds a read-only SAS for the container. The SAS
itself never appears here.

In [ ]:
ACCOUNT_URL = "https://<storage-account>.blob.core.windows.net"
CONTAINER = "<container>"
ROOT = "<folder>"
SAS_SECRET_NAME = "<secret-name>"

<a id="connect"></a>
## 3. Connect

Connecting validates the dataset, so connect while it is clean. The cell lists the datasets already
connected and reuses the one that points at this folder, so re-running it never connects twice.

In [ ]:
connected = conn.get(path="dataset")["datasets"]
display(pd.DataFrame(connected, columns=["id", "name", "layout", "location", "created_at"]))

location = f"{ACCOUNT_URL}/{CONTAINER}/{ROOT}".rstrip("/")
DATASET_ID = next((d["id"] for d in connected if d["location"].rstrip("/") == location), None)

In [ ]:
if DATASET_ID:
    print("Already connected:", DATASET_ID)
else:
    try:
        dataset = conn.post(path="dataset", data={
            "name": ROOT,
            "source": {
                "url": f"{ACCOUNT_URL}/{CONTAINER}",
                "root": ROOT,
                "credential": {"kind": "key_vault_secret", "name": SAS_SECRET_NAME},
            },
            "format": {"layout": "lerobot"},
        })
        DATASET_ID = dataset["id"]
        print("Connected:", DATASET_ID)
    except InvalidRequestError as error:
        summarize_validation(error=error)

<a id="validate"></a>
## 4. Validate

Checks the layout: the manifest, the episode index, and that every file they name is there.

In [ ]:
try:
    validation = conn.post(path=f"dataset/{DATASET_ID}/validate")
    summarize_validation(validation)
except InvalidRequestError as error:
    summarize_validation(error=error)

<a id="audit"></a>
## 5. Audit

Tick the checks to run in the boxes below; none are ticked to start with. The audit decodes
every episode once and runs the ticked checks:

- **Files against metadata**: `completeness` (every frame the metadata promises is there) and
  `camera_validity` (each camera's video decodes and matches its declared format).
- **What the footage shows**: `dark_intervals`, `blur`, `static_footage` (a frozen picture) and
  `shot_cuts` (the view jumps).
- **The robot's numbers**: `numeric_integrity` (no missing or NaN values), `abrupt_transitions`
  (a joint jumps between steps) and `statistical_outliers` (values outside the range the audited
  episodes share).

A gripper opens and shuts in one step by design, so `EXCLUDE` leaves its coordinate out of the jump
and range checks; index 5 is the SO-101 gripper. Set `EPISODES` to a list, such as `[0, 2]`, to audit
only those episodes; the range check compares the episodes audited together.

Change the ticks and re-run only the **Audit** cell. The table lists each problem once. Everything Curation returned stays in `audit` and `episodes`.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

CHECK_GROUPS = {
    "Files against metadata": ["completeness", "camera_validity"],
    "What the footage shows": ["dark_intervals", "blur", "static_footage", "shot_cuts"],
    "The robot's numbers": ["numeric_integrity", "abrupt_transitions", "statistical_outliers"],
}
CHECK_BOXES = {check: widgets.Checkbox(value=False, description=check, indent=False)
               for checks in CHECK_GROUPS.values() for check in checks}
display(widgets.HBox([
    widgets.VBox([widgets.HTML(f"<b>{group}</b>"), *(CHECK_BOXES[c] for c in checks)])
    for group, checks in CHECK_GROUPS.items()
], layout=widgets.Layout(gap="32px")))

In [ ]:
EPISODES = "all"
CHECKS = [check for check, box in CHECK_BOXES.items() if box.value]
EXCLUDE = {"action": [5], "observation.state": [5]}  # SO-101 gripper

if not CHECKS:
    print("Tick at least one check above, then re-run this cell.")
else:
    job = conn.post(path=f"dataset/{DATASET_ID}/audit",
                    data={"episodes": EPISODES, "checks": CHECKS, "exclude_coordinates": EXCLUDE})
    while (result := conn.session.get(job["output_url"], timeout=30).json())["status"] in ("processing", "in progress"):
        time.sleep(5)
    audit = result["response"]

    if result["status"] != "complete":
        print("❌ The audit did not finish:", audit.get("message", audit))
    else:
        episodes = conn.get(path=f"dataset/{DATASET_ID}/audit/{job['id']}/episodes", params={"limit": 500})["episodes"]
        summarize_audit(episodes, CHECKS)

<a id="disconnect"></a>
## 6. Disconnect (optional)

Forgets the connection. Curation never changes a connected dataset's files.

In [ ]:
# conn.delete(path=f"dataset/{DATASET_ID}")